Generally, this script will standardize the outputs across tools by
- (1) reading in all local footprinting results
- (2) ensure motif names are consistent
- (3) ensure peak-coordinates and footprint-coordinates are saved
- (4) transform into a pivot table of identical dimensions and order

## Table of Contents:
* [1. HINT](#first-bullet)
* [2. TOBIAS](#second-bullet)
* [3. PRINT](#third-bullet)
* [4. PWM](#fourth-bullet)

In [ ]:
! wc -l ../05_footprinting/01_original/hint/*_mpbs.bed
#  11599636 GM12878-cellFilt_mpbs.bed
#   8603008 HEPG2-cellFilt_mpbs.bed
#   8839956 K562-cellFilt_mpbs.bed
#   7949708 MCF7-cellFilt_mpbs.bed
#   7700362 SKNSH-cellFilt_mpbs.bed

! wc -l ../05_footprinting/01_original/tobias/*_bindetect_TF_overviews.txt
#   90970153 GM12878-cellFilt_bindetect_TF_overviews.txt
#   74512072 HEPG2-cellFilt_bindetect_TF_overviews.txt
#   79882033 K562-cellFilt_bindetect_TF_overviews.txt
#   70647796 MCF7-cellFilt_bindetect_TF_overviews.txt
#   65795293 SKNSH-cellFilt_bindetect_TF_overviews.txt

! wc -l ../05_footprinting/01_original/print/*_granges.bed
#   21022707 GM12878_granges.bed
#   18809955 HEPG2_granges.bed
#   21124596 K562_granges.bed
#   17103002 MCF7_granges.bed
#   18935463 SKNSH_granges.bed

In [ ]:
import pybedtools, os, re
import pandas as pd
import numpy as np
from pathlib import Path

def FindMissing(data_dir, out_dir, prefix):
    in_files = [x for x in os.listdir(data_dir) if x.endswith(prefix) ]  
    out_files = [x for x in os.listdir(out_dir) if x.endswith(".mat") ]
    out_names = [re.sub(".mat", prefix, f_foot, count=1) for f_foot in out_files]
    left = list(set(in_files).difference(set(out_names)))
    return left


all_dirs = ["01_original", "02_cells", "03_reads", "04_frip"] #05_peakreads

tool = "hint"
for dd in all_dirs:
    out_dir = f"../06_dataquality/{tool}/"
    data_dir = f"../05_footprinting/{dd}/{tool}"
    all_files  = FindMissing(data_dir, f"{out_dir}/mats/", "_mpbs.bed")
    if len(all_files) != 0 : print(f"Missing the mat files for {tool}:{dd}\n{all_files}")


tool = "tobias"
for dd in all_dirs:
    out_dir = f"../06_dataquality/{tool}/"
    data_dir = f"../05_footprinting/{dd}/{tool}"
    all_files  = FindMissing(data_dir, f"{out_dir}/mats/bound_threshold/", "_bindetect_TF_overviews.txt")
    all_files  = FindMissing(data_dir, f"{out_dir}/mats/static_threshold/", "_bindetect_TF_overviews.txt")
    if len(all_files) != 0 : print(f"Missing the mat files for {tool}:{dd}\n{all_files}")


tool = "print"
for dd in all_dirs:
    out_dir = f"../06_dataquality/{tool}/"
    data_dir = f"../05_footprinting/{dd}/{tool}"
    all_files  = FindMissing(data_dir, f"{out_dir}/mats/thresh_03/", "_granges.bed")
    all_files  = FindMissing(data_dir, f"{out_dir}/mats/thresh_05/", "_granges.bed")
    if len(all_files) != 0 : print(f"Missing the mat files for {tool}:{dd}\n{all_files}")

print("done")

# HINT <a class="anchor" id="first-bullet"></a>

In [ ]:
###   HINT  ###
import pybedtools, os, re
import pandas as pd
import numpy as np
from pathlib import Path

out_dir = "../06_dataquality/hint/"
#data_dir = "../05_footprinting/01_original/hint"
#data_dir = "../05_footprinting/02_cells/hint"
#data_dir = "../05_footprinting/03_reads/hint"
data_dir = "../05_footprinting/04_frip/hint"

output_cleaned_beds = False
if output_cleaned_beds: Path(f"{out_dir}/cleaned/").mkdir(parents=True, exist_ok=True)
Path(f"{out_dir}/mats/").mkdir(parents=True, exist_ok=True)
    
all_files = [x for x in os.listdir(data_dir) if x.endswith("_mpbs.bed") ]
cell_lines = ["K562", "HEPG2", "GM12878", "MCF7", "SKNSH"]

all_files  = FindMissing(data_dir, f"{out_dir}/mats/", "_mpbs.bed")
cell_lines = [v1 for v1 in cell_lines if any(v1 in v2 for v2 in all_files)]

for cl in cell_lines:
    files = [f for f in all_files if cl in f]
    
    #get all the atac_regions
    peak_file = f"../03_peakcalls/{cl}_filt_500bp.exclusion.bed"
    df = pd.read_csv(f"{peak_file}", header=None, sep="\t")
    df["region_atac"] = df[0].astype(str) + ":" + df[1].astype(str) + "-" + df[2].astype(str)
    all_atac_regions = df["region_atac"].tolist()
    
    #get all the tfs and add a way to standardize the names. 
    infile="../05_footprinting/program_input_files/jaspar2022_motifs_to_clusters.txt"
    TF_df = pd.read_csv(infile, sep=" ", header=None, usecols = [0], names=["motif"])
    TF_df["motif"] = [i.upper() for i in TF_df["motif"]] 
    TF_df["hint_convert"] = [i.split('_')[1]+"."+i.split('_')[0] for i in TF_df["motif"]]
    all_tfs = TF_df["motif"].tolist()
    
    #for all files and threshold combos, make the matrices. 
    for f_foot in files:
        name = re.sub('_mpbs.bed', '', f_foot, count=1)
        print(f"working on: {name}")
        
        #interesct the two files so we have the tag counts. 
        file2 = re.sub('_mpbs.bed', '.bed', f_foot, count=1)
        ! bedtools intersect \
            -a <(awk '{{print $$1,$$2,$$3,$$5}}' {data_dir}/{file2} | tr " " "\t") \
            -b <(awk '{{sub(/[ \t]+$$/, ""); print}}' {data_dir}/{f_foot} | sort -k1,1 -k2,2n) \
            -F 1 -wb -wa -sorted | awk '{{print $$5,$$6,$$7,$$8,$$9,$$4}}' > current_hint.bed

        #intersect again with our peak file.
        #note the 50% here is because hint ft can overhang the peak regions. 
        #interestingly there are a few that fall very near but technically outside. it must be 
        #part of how the original bed files are constructed. 
        #so the numbers between mpbs files and these beds will be slightly off. 
        ! bedtools intersect \
            -a {peak_file} \
            -b <(cat current_hint.bed | tr " " "\t") \
            -F 0.5 -wb -wa -sorted > current_hint2.bed

        #read in df and merge w/ tag count. 
        df = pd.read_csv("current_hint2.bed", sep="\t", header=None)
        df = df.drop_duplicates()
        df["region_atac"] = df[0].astype(str) + ":" + df[1].astype(str) + "-" + df[2].astype(str)
        df["region_TFBS"] = df[6].astype(str) + ":" + df[7].astype(str) + "-" + df[8].astype(str)
        df["curr_motif"] = [i.upper() for i in df[9]]
        df["pwm_score"] = df[10]
        df["tag_count"] = df[11]
        df1 = df.merge(TF_df, right_on='hint_convert', left_on="curr_motif", how='left')

        if output_cleaned_beds:
            df1[["region_atac", "region_TFBS", "motif","pwm_score","tag_count"]].to_csv(f"{out_dir}/cleaned/{name}.bed", sep="\t", index=False)
            
        df_observed = df1.pivot_table(index="region_atac", columns="motif", aggfunc='size', fill_value=0)
        df_observed = df_observed.reindex(all_tfs, axis=1, fill_value=0)
        df_observed = df_observed.reindex(all_atac_regions, axis=0, fill_value=0)
        
        if df.shape[0] == df_observed.sum().sum():
            df_observed.to_csv(f"{out_dir}/mats/{name}.mat", index=True)
        else:
            print("ERRORRRRR")


# TOBIAS <a class="anchor" id="second-bullet"></a>

In [ ]:
import pybedtools, os, re
import pandas as pd
import numpy as np
from pathlib import Path

out_dir = "../06_dataquality/tobias/"
data_dir = "../05_footprinting/01_original/tobias"
Path(f"{out_dir}/cleaned/").mkdir(parents=True, exist_ok=True)

all_files = [x for x in os.listdir(data_dir) if x.endswith("_bindetect_TF_overviews.txt") ]
cell_lines = ["HEPG2", "MCF7", "SKNSH", "MCF7"]

for cl in cell_lines:
    print(cl)
    files = [f for f in all_files if cl in f]
    
    #get all the tfs and add a way to standardize the names. 
    infile="../05_footprinting/program_input_files/jaspar2022_motifs_to_clusters.txt"
    TF_df = pd.read_csv(infile, sep=" ", header=None, usecols = [0], names=["motif"])
    TF_df["motif"] = [i.upper() for i in TF_df["motif"]] 
    TF_df["tobias_convert"] = [re.sub('::', '', i) for i in TF_df["motif"]]
    all_tfs = TF_df["motif"].tolist()
    
    #for all files and threshold combos, make the matrices. 
    for f_foot in files:
        name = re.sub('_bindetect_TF_overviews.txt', '', f_foot, count=1)
        print(f"working on: {name}")
        
        cmd = f'''awk '{{split($4, a, "[_]"); $4 = a[1]"_"a[2]; print $7":"$8"-"$9, $1":"$2"-"$3, toupper($4), $13, $14}}' {data_dir}/{f_foot} > current_tobias.bed'''
        ! {cmd}
        
        df = pd.read_csv("current_tobias.bed", sep=" ", names=["region_atac", "region_TFBS", "curr_motif", "score", "boolean_score"])
        df1 = df.merge(TF_df, right_on='tobias_convert', left_on="curr_motif", how='left')
        
        #df1[["region_atac", "region_TFBS", "motif", "score"]].to_csv(f"{out_dir}/cleaned/{name}.bed", sep="\t", index=False)
        df1[["region_atac", "region_TFBS", "motif", "score", "boolean_score"]].to_csv(f"{out_dir}/cleaned/{name}.bed", sep="\t", index=False)

        

In [ ]:
import pybedtools, os, re
import pandas as pd
import numpy as np
from pathlib import Path

out_dir = "../06_dataquality/tobias/"
data_dir = "../05_footprinting/02_cells/tobias"
#data_dir = "../05_footprinting/03_reads/tobias"
#data_dir = "../05_footprinting/04_frip/tobias"

matdir = f"{out_dir}/mats/bound_threshold/"
Path(matdir).mkdir(parents=True, exist_ok=True)

all_files = [x for x in os.listdir(data_dir) if x.endswith("_bindetect_TF_overviews.txt") ]
cell_lines = ["K562", "HEPG2", "GM12878", "MCF7", "SKNSH"]

all_files  = FindMissing(data_dir, mat_dir, "_bindetect_TF_overviews.txt")
cell_lines = [v1 for v1 in cell_lines if any(v1 in v2 for v2 in all_files)]

for cl in cell_lines:
    files = [f for f in all_files if cl in f]
    
    #get all the atac_regions
    peak_file = f"../03_peakcalls/{cl}_filt_500bp.exclusion.bed"
    df = pd.read_csv(f"{peak_file}", header=None, sep="\t")
    df["region_atac"] = df[0].astype(str) + ":" + df[1].astype(str) + "-" + df[2].astype(str)
    all_atac_regions = df["region_atac"].tolist()
    
    #get all the tfs and add a way to standardize the names. 
    infile="../05_footprinting/program_input_files/jaspar2022_motifs_to_clusters.txt"
    TF_df = pd.read_csv(infile, sep=" ", header=None, usecols = [0], names=["motif"])
    TF_df["motif"] = [i.upper() for i in TF_df["motif"]] 
    TF_df["tobias_convert"] = [re.sub('::', '', i) for i in TF_df["motif"]]
    all_tfs = TF_df["motif"].tolist()
    
    #for all files and threshold combos, make the matrices. 
    for f_foot in files:
        name = re.sub('_bindetect_TF_overviews.txt', '', f_foot, count=1)
        print(f"working on: {name}")
        
        #make a file we can work with
        cmd = f'''awk '$14==1 {{split($4, a, "[_]"); $4 = a[1]"_"a[2]; print $7":"$8"-"$9, $1":"$2"-"$3, toupper($4), $13}}' {data_dir}/{f_foot} > current_tobias.bed'''
        ! {cmd}
        
        df = pd.read_csv("current_tobias.bed", sep=" ", names=["region_atac", "region_TFBS", "curr_motif", "score"])
        df1 = df.merge(TF_df, right_on='tobias_convert', left_on="curr_motif", how='left')
 
        df_observed = df1.pivot_table(index="region_atac", columns="motif", aggfunc='size', fill_value=0)
        df_observed = df_observed.reindex(all_tfs, axis=1, fill_value=0)
        df_observed = df_observed.reindex(all_atac_regions, axis=0, fill_value=0)
        
        if df.shape[0] == df_observed.sum().sum():
            df_observed.to_csv(f"{matdir}/{name}.mat", index=True)
        else:
            print("ERRORRRRR")

In [ ]:
import pybedtools, os, re
import pandas as pd
import numpy as np
from pathlib import Path

out_dir = "../06_dataquality/tobias/"
data_dir = "../05_footprinting/01_original/tobias"

matdir = f"{out_dir}/mats/static_threshold/"
Path(matdir).mkdir(parents=True, exist_ok=True)

all_files = [x for x in os.listdir(data_dir) if x.endswith("_bindetect_TF_overviews.txt") ]
thresh_dict = {"K562":0.13426, "GM12878":0.02951, "HEPG2":0.18492, "MCF7":0.19234, "SKNSH":0.16365}

all_files  = FindMissing(data_dir, matdir, "_bindetect_TF_overviews.txt")
print(all_files)
cell_lines = [cl for cl in list(thresh_dict.keys()) if any(cl in f for f in all_files)]

print(cell_lines)
for cl in cell_lines:
    files = [f for f in all_files if cl in f]
    tobias_bound = thresh_dict[cl]
    
    #get all the atac_regions
    peak_file = f"../03_peakcalls/{cl}_filt_500bp.exclusion.bed"
    df = pd.read_csv(f"{peak_file}", header=None, sep="\t")
    df["region_atac"] = df[0].astype(str) + ":" + df[1].astype(str) + "-" + df[2].astype(str)
    all_atac_regions = df["region_atac"].tolist()
    
    #get all the tfs and add a way to standardize the names. 
    infile="../05_footprinting/program_input_files/jaspar2022_motifs_to_clusters.txt"
    TF_df = pd.read_csv(infile, sep=" ", header=None, usecols = [0], names=["motif"])
    TF_df["motif"] = [i.upper() for i in TF_df["motif"]] 
    TF_df["tobias_convert"] = [re.sub('::', '', i) for i in TF_df["motif"]]
    all_tfs = TF_df["motif"].tolist()
    
    #for all files and threshold combos, make the matrices. 
    for f_foot in files:
        name = re.sub('_bindetect_TF_overviews.txt', '', f_foot, count=1)
        print(f"working on: {name}")
        
        #make a file we can work with
        cmd = f'''awk '$13> {tobias_bound} {{split($4, a, "[_]"); $4 = a[1]"_"a[2]; print $7":"$8"-"$9, $1":"$2"-"$3, toupper($4), $13}}' {data_dir}/{f_foot} > current_tobias.bed'''
        ! {cmd}
        
        df = pd.read_csv("current_tobias.bed", sep=" ", names=["region_atac", "region_TFBS", "curr_motif", "score"])
        df1 = df.merge(TF_df, right_on='tobias_convert', left_on="curr_motif", how='left')
 
        df_observed = df1.pivot_table(index="region_atac", columns="motif", aggfunc='size', fill_value=0)
        df_observed = df_observed.reindex(all_tfs, axis=1, fill_value=0)
        df_observed = df_observed.reindex(all_atac_regions, axis=0, fill_value=0)
        
        if df.shape[0] == df_observed.sum().sum():
            df_observed.to_csv(f"{matdir}/{name}.mat", index=True)
        else:
            print("ERRORRRRR")

# PRINT <a class="anchor" id="fourth-bullet"></a>

In [ ]:
import pybedtools, os, re
import pandas as pd
import numpy as np
from pathlib import Path

out_dir = "../06_dataquality/print/"
data_dir = "../05_footprinting/01_original/print/"; output_cleaned_beds = True

all_files = [x for x in os.listdir(data_dir) if x.endswith("_granges.bed") ]
cell_lines = ["K562", "HEPG2", "GM12878", "MCF7", "SKNSH"]

scores = [] 
for cl in cell_lines:
    files = [f for f in all_files if cl in f]

    #for all files and threshold combos, make the matrices. 
    for f_foot in files:
        #name = re.sub('_granges.bed', '', f_foot, count=1)
        name = re.sub('_granges.bed', '-cellFilt', f_foot, count=1)
        print(name)
    
        df = pd.read_csv(f"{out_dir}/cleaned/{name}.bed", sep="\t")
        scores.append(df["score"].values)
        
final_array = np.concatenate(scores, axis=0)
print("done")

In [ ]:
import pybedtools, os, re
import pandas as pd
import numpy as np
from pathlib import Path

out_dir = "../06_dataquality/print/"
#data_dir = "../05_footprinting/01_original/print/"; output_cleaned_beds = True
#data_dir = "../05_footprinting/02_cells/print/"; output_cleaned_beds = False
#data_dir = "../05_footprinting/03_reads/print/"; output_cleaned_beds = False
data_dir = "../05_footprinting/04_frip/print/"; output_cleaned_beds = False

if output_cleaned_beds: Path(f"{out_dir}/cleaned/").mkdir(parents=True, exist_ok=True)
Path(f"{out_dir}/mats/").mkdir(parents=True, exist_ok=True)

all_files = [x for x in os.listdir(data_dir) if x.endswith("_granges.bed") ]
cell_lines = ["K562", "HEPG2", "GM12878", "MCF7", "SKNSH"]

all_files  = FindMissing(data_dir, f"{out_dir}/mats/", "_granges.bed")
cell_lines = [v1 for v1 in cell_lines if any(v1 in v2 for v2 in all_files)]

for cl in cell_lines:
    files = [f for f in all_files if cl in f]
    
    #get all the atac_regions
    peak_file = f"../03_peakcalls/{cl}_filt_500bp.exclusion.bed"
    df = pd.read_csv(f"{peak_file}", header=None, sep="\t")
    df["region_atac"] = df[0].astype(str) + ":" + df[1].astype(str) + "-" + df[2].astype(str)
    all_atac_regions = df["region_atac"].tolist()
    
    #get all the tfs and add a way to standardize the names. 
    infile="../05_footprinting/program_input_files/jaspar2022_motifs_to_clusters.txt"
    TF_df = pd.read_csv(infile, sep=" ", header=None, usecols = [0], names=["motif"])
    TF_df["motif"] = [i.upper() for i in TF_df["motif"]] 
    all_tfs = TF_df["motif"].tolist()
    
    #for all files and threshold combos, make the matrices. 
    for f_foot in files:
        name = re.sub('_granges.bed', '', f_foot, count=1)
        #name = re.sub('_granges.bed', '-cellFilt', f_foot, count=1)
        print(f"working on: {name}")

        #intersect with our peak file.
        #this is really overkill since all the files are in the same order, but leaving it for consistencies sake. 
        ! bedtools intersect \
            -a {peak_file} \
            -b <(tail -n+2 {data_dir}{f_foot} | tr " " "\t" | sort -k1,1 -k2,2n) \
            -F 0.5 -wb -wa -sorted > current_print2.bed

        #read in df and merge by tag count. 
        df = pd.read_csv("current_print2.bed", sep="\t", header=None) 
        df = df.drop_duplicates()
        df["region_atac"] = df[0].astype(str) + ":" + df[1].astype(str) + "-" + df[2].astype(str)
        df["region_TFBS"] = df[6].astype(str) + ":" + df[7].astype(str) + "-" + df[8].astype(str)
        df["motif"] = [i.upper() for i in df[12]]
        df["score"] = df[13]
        
        if output_cleaned_beds:
            df[["region_atac", "region_TFBS", "motif","score"]].to_csv(f"{out_dir}/cleaned/{name}.bed", sep="\t", index=False)
                
        df = df[df["score"] >= 0.3]
        df_observed = df.pivot_table(index="region_atac", columns="motif", aggfunc='size', fill_value=0)
        df_observed = df_observed.reindex(all_tfs, axis=1, fill_value=0)
        df_observed = df_observed.reindex(all_atac_regions, axis=0, fill_value=0)
        
        if df.shape[0] == df_observed.sum().sum():
            df_observed.to_csv(f"{out_dir}/mats/{name}.mat", index=True)
        else:
            print("ERRORRRRR")
        

# PWM <a class="anchor" id="third-bullet"></a>

In [ ]:
import pybedtools, os, re
import pandas as pd
import numpy as np
from pathlib import Path

def adjust_index(index):
    import re
    match = re.match(r"([^:]+):(\d+)-(\d+)", index)
    if match:
        chrom, start, end = match.groups()
        return f"{chrom}:{int(start)-1}-{end}"
    return index 

out_dir = "../06_dataquality/pwm/"
data_dir = "../05_footprinting/01_original/pwm"

output_cleaned_beds = True
if output_cleaned_beds: Path(f"{out_dir}/cleaned/").mkdir(parents=True, exist_ok=True)
Path(f"{out_dir}/mats/").mkdir(parents=True, exist_ok=True)

all_files = [x for x in os.listdir(data_dir) if x.endswith("_PWMmatch.bed") ]
cell_lines = ["K562", "HEPG2", "GM12878", "MCF7", "SKNSH"]

for cl in cell_lines:
    files = [f for f in all_files if cl in f]
    
    #get all the atac_regions
    peak_file = f"../03_peakcalls/{cl}_filt_500bp.exclusion.bed"
    df = pd.read_csv(f"{peak_file}", header=None, sep="\t")
    df["region_atac"] = df[0].astype(str) + ":" + df[1].astype(str) + "-" + df[2].astype(str)
    all_atac_regions = df["region_atac"].tolist()
    
    #get all the tfs and add a way to standardize the names. 
    infile="../05_footprinting/program_input_files/jaspar2022_motifs_to_clusters.txt"
    TF_df = pd.read_csv(infile, sep=" ", header=None, usecols = [0], names=["motif"])
    TF_df["motif"] = [i.upper() for i in TF_df["motif"]] 
    all_tfs = TF_df["motif"].tolist()
    
    #for all files and threshold combos, make the matrices. 
    for f_foot in files:
        #name = re.sub('_PWMmatch.bed', '.mat', f_foot, count=1)
        name = re.sub('_filt_500bp.exclusion_PWMmatch.bed', '-cellFilt', f_foot, count=1) #original to make it match
        print(f"working on: {name}")
        
        df = pd.read_csv(f"{data_dir}/{f_foot}", sep="\t") 
        df.columns = ["region_TFBS", "region_atac", "motif","score"]
        df.motif = [("_".join(i.split("_", 2)[:2])).upper() for i in df["motif"]]
        df.region_atac = df.region_atac.map(adjust_index)
        
        if output_cleaned_beds:
            df[["region_atac", "region_TFBS", "motif","score"]].to_csv(f"{out_dir}/cleaned/{name}.bed", sep="\t", index=False)
                
        df_observed = df.pivot_table(index="region_atac", columns="motif", aggfunc='size', fill_value=0)
        df_observed = df_observed.reindex(all_tfs, axis=1, fill_value=0)
        df_observed = df_observed.reindex(all_atac_regions, axis=0, fill_value=0)
        
        if df.shape[0] == df_observed.sum().sum():
            df_observed.to_csv(f"{out_dir}/mats/{name}.mat", index=True)
        else:
            print("ERRORRRRR")